In [25]:
import dotenv
import os

dotenv.load_dotenv()

True

In [ ]:
import pandas as pd
from langchain_core.documents import Document

data = pd.read_csv("dataset/Tamil_movies_dataset.csv")

def generate_movie_profile(row):
    return (
        f"Title: {row['MovieName']}\n"
        f"Genre: {row['Genre']}\n"
        f"Director: {row['Director']}\n"
        f"Actor: {row['Actor']}\n"
        f"Release Year: {row['Year']}\n"
        f"Rating: {row['Rating']}\n"
    )

data["combinedrating"]=data.apply(generate_movie_profile, axis=1)

documents = [
    Document(
        page_content=row["combinedrating"],
    )
]




In [27]:
from pydantic import BaseModel,Field
from typing import List

class Shot(BaseModel):
    shotnumber:int
    visual:str=Field(description="A brief description of the visual content of the shot.")
    cameraangle:str=Field(description="The camera angle used in the shot, e.g., close-up, wide shot, aerial view.")
    audio_cue:str=Field(description="Any significant audio cues present in the shot, such as dialogue, sound effects, or music.")

class TrailerPackage(BaseModel):
    structure: str = Field(description="The 3-act breakdown of the trailer")
    voice_over: str = Field(description="The script for the narrator")
    music_mood: str = Field(description="Instrumentation, tempo, and vibe")
    fonrstyle: str = Field(description="The font style to be used in the trailer")
    title: str = Field(description="The title of the movie in tamil and english that is good make it catchy and appealing")
    shot_list: List[Shot]


In [28]:
from langchain_core.prompts import ChatPromptTemplate

prompt =ChatPromptTemplate.from_template(
    """
    You are a Kollywood Trailer Editor who is an expert in creating engaging and captivating trailers for Tamil movies.
    Your task is to analyze the provided movie plot and generate a detailed trailer structure that includes a 3-act breakdown, voice-over script, music mood, and a shot list with descriptions of visuals, camera angles, and audio cues using the synopsis of the movie
    MOVIE SYNOPSIS: {synopsis}
    ADDITIONAL STYLE GUIDELINES: {instructions}
    """
)

In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.getenv("GEMINI_API_KEY"),temperature=0.7)

def generate_trailer_package(payload: dict) -> TrailerPackage:
    structeredllm=model.with_structured_output(TrailerPackage)
    synopsis = payload.get("synopsis")
    user_instructions = payload.get("user_instructions", None)
    chain=prompt|structeredllm
    response= chain.invoke({"synopsis": synopsis,"instructions": user_instructions if user_instructions else "No specific style requested."})
    data = response.model_dump()
    
    for key, value in data.items():
        title = key.replace("_", " ").upper()
        print(f"{title}")
        if isinstance(value, list):
            for item in value:
                shot_num = item.get('shot_number', '-')
                print(f"Shot {shot_num}: {item.get('visual')}")
                print(f"Camera Angle: {item.get('cameraangle')}")
                print(f"Audio Cue: {item.get('audio_cue')}\n")
        else:
            print(f"{value}\n")
            
    return response

   

    

In [39]:
repsone=generate_trailer_package({"synopsis":"""
A fearless officer investigates a string of brutal murders, only to uncover a chilling motive behind the serial killer’s deadly game.
""","user_instructions":"Make it dark and thrilling with a touch of suspense."})  

STRUCTURE
ACT 1: THE SHADOW DESCENDS (0:00 - 0:45) - Introduces the city under a wave of brutal murders, the fear gripping its inhabitants, and the entry of a fearless officer determined to crack the case. Establishes the initial mystery and the high stakes. ACT 2: THE HUNT DEEPENS (0:45 - 1:30) - The investigation escalates. The officer uncovers cryptic clues, faces dead ends, and realizes the killer's methods are more complex and chilling than initially thought. Tension mounts as the officer gets closer to a dark truth. ACT 3: THE CHILLING REVELATION (1:30 - 2:00) - A rapid montage of intense action, confrontations, and a terrifying glimpse into the killer's motive. The trailer culminates in a powerful reveal, leaving the audience on the edge of their seats.

VOICE OVER
இந்த நகரத்தை ஒரு இருண்ட நிழல் சூழ்ந்துள்ளது. (A dark shadow has engulfed this city.) ஒவ்வொரு இரவும், ஒரு புதிய பயங்கரம். (Every night, a new horror.) யார் இந்த கொடூரங்களை செய்கிறார்கள்? (Who commits these atrocities?)